# Funciones usadas durante el proceso

## Función para el cruce entre tablas de entrada y Códigos Territoriales

In [0]:
def cruce_codigos_territoriales(df, df_ct):
  df_cruce_reg = df.alias("v").join(
    broadcast(df_ct.alias("ct")), 
    [col("v.region") == col("ct.codigo_territorial"), col("ct.division_politica") == "Región"],
    "inner")

  # Elimina campos originales y resultados del cruce que no son necesarios
  df_cruce_reg = df_cruce_reg.drop("region", "division_politica", "codigo_territorial")
  # Renombra el campo 'territorio'
  df_cruce_reg = df_cruce_reg.withColumnRenamed("territorio", "region")

  df_cruce_prov = df_cruce_reg.alias("cr").join(
      broadcast(df_ct.alias("ct")), 
      [col("cr.provincia") == col("ct.codigo_territorial"), col("ct.division_politica") == "Provincia"], 
      "inner")

  # Elimina campos originales y resultados del cruce que no son necesarios
  df_cruce_prov = df_cruce_prov.drop("provincia", "division_politica", "codigo_territorial")
  # Renombra el campo 'territorio'
  df_cruce_prov = df_cruce_prov.withColumnRenamed("territorio", "provincia")

  df_cruce = df_cruce_prov.alias("cp").join(
      broadcast(df_ct.alias("ct")), 
      [col("cp.comuna") == col("ct.codigo_territorial"), col("ct.division_politica") == "Comuna"], 
      "inner")

  # Elimina campos originales y resultados del cruce que no son necesarios
  df_cruce = df_cruce.drop("comuna", "division_politica", "codigo_territorial")
  # Renombra el campo 'territorio'
  df_cruce = df_cruce.withColumnRenamed("territorio", "comuna")

  return df_cruce


## Función para el cruce entre tablas de entrada y Códigos Varios

In [0]:
def cruce_codigos_otros(df, df_co, cols_gb):
    df_long = df.select(*cols_gb,
        explode(
            array([
                struct(lit(c).alias("campo"), col(c).cast("string").alias("codigo")) 
                for c in df.columns
            ])
        ).alias("kv")
    ).select(*cols_gb, col("kv.campo").alias("co_campo"), col("kv.codigo"))


    joined = df_long.join(
        broadcast(df_co),
        on=[df_long.co_campo == df_co.campo, df_long.codigo == df_co.codigo],
        how="left"
    ) .withColumn("descripcion", \
            when(df_co.descripcion.isNull(), df_long.codigo) \
            # No respuesta. Dejo null.
            .when(df_long.codigo == "-99", lit(None)) \
            # Valor suprimido por anonimizacion. Dejo null.
            .when(df_long.codigo == "-66", lit(None))
            .otherwise(df_co.descripcion)) \
            .select(*cols_gb, df_long.co_campo, "descripcion")

    df_desc = joined \
            .groupBy(*cols_gb) \
            .pivot("co_campo") \
            .agg(first("descripcion"))

    # Filtro los campos
    # Si existen en df_desc tomo ese, si no existe tomo el del DF original
    # ya que df_desc tiene los valores de los codigos
    cols_final = [col(f"df_desc.{c}") for c in df.columns if c in df_desc.columns] + [col(f"df.{c}") for c in df.columns if c not in df_desc.columns]

    # Recupero todos los campos 
    df_final = df.alias("df").join(df_desc.alias("df_desc"), on=cols_gb, how="inner").select(*cols_final)

    return df_final